# PointMAE — Self-Supervised LiDAR Sensor Encoder
**GM AI/ML Intern Project | Rithvik Vadlamani**

This notebook trains a Masked Autoencoder on 3D point clouds using the free **ModelNet10** dataset.
No account or payment needed. Just click **Runtime → Run All**.

**Pipeline:**
1. Download ModelNet10 (free, ~50MB)
2. Build PointMAE model (5.5M params)
3. Train with self-supervised masking
4. Visualise loss curve + reconstructed patches

In [ ]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────────
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('CUDA:', torch.version.cuda)

In [ ]:
# ── 1. Clone repo & install deps ─────────────────────────────────────────────
!git clone https://github.com/Rithvik-Vadlamani/lidar-sensor-encoder.git
%cd lidar-sensor-encoder
!pip install torch torchvision numpy pyyaml matplotlib -q

In [ ]:
# ── 2. Download ModelNet10 (free, no login) ───────────────────────────────────
import os, zipfile, urllib.request

URL  = 'http://3dvision.princeton.edu/projects/2014/3DShapeNets/ModelNet10.zip'
DEST = 'ModelNet10.zip'

if not os.path.exists('ModelNet10'):
    print('Downloading ModelNet10...')
    urllib.request.urlretrieve(URL, DEST)
    print('Extracting...')
    with zipfile.ZipFile(DEST, 'r') as z:
        z.extractall('.')
    os.remove(DEST)
    print('Done.')
else:
    print('Already downloaded.')

# List classes
classes = sorted(os.listdir('ModelNet10'))
print(f'Classes ({len(classes)}):', classes)

In [ ]:
# ── 3. Dataset: load .off mesh files → sample point clouds ───────────────────
import numpy as np
import glob
from torch.utils.data import Dataset, DataLoader

def load_off(filepath):
    """Parse .off 3D mesh file and return vertices as (N,3) numpy array."""
    with open(filepath, 'r') as f:
        lines = f.read().splitlines()
    # Handle OFF header on same line as counts
    start = 1
    if lines[0].strip() != 'OFF':
        counts = lines[0][3:].split()
    else:
        counts = lines[1].split()
        start = 2
    n_verts = int(counts[0])
    verts = []
    for line in lines[start:start + n_verts]:
        verts.append([float(x) for x in line.split()])
    return np.array(verts, dtype=np.float32)

class ModelNet10Dataset(Dataset):
    def __init__(self, root='ModelNet10', split='train', n_points=512):
        self.n_points = n_points
        self.files = []
        for cls in sorted(os.listdir(root)):
            pattern = os.path.join(root, cls, split, '*.off')
            self.files.extend(glob.glob(pattern))
        print(f'  {split}: {len(self.files)} meshes found')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        verts = load_off(self.files[idx])
        # Random subsample to fixed size
        if len(verts) >= self.n_points:
            idx2 = np.random.choice(len(verts), self.n_points, replace=False)
        else:
            idx2 = np.random.choice(len(verts), self.n_points, replace=True)
        pts = verts[idx2]
        # Normalize to unit sphere
        pts -= pts.mean(axis=0)
        pts /= (np.abs(pts).max() + 1e-8)
        return torch.from_numpy(pts).float()

print('Loading datasets...')
train_ds = ModelNet10Dataset(split='train', n_points=512)
test_ds  = ModelNet10Dataset(split='test',  n_points=512)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# ── 4. Build model ────────────────────────────────────────────────────────────
from models import PointMAE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training on:', device)

model = PointMAE(
    n_centers=32, k=16, embed_dim=256,
    enc_n_heads=4, enc_n_layers=6, enc_ffn_dim=512,
    dec_n_heads=4, dec_n_layers=4, dec_ffn_dim=512,
    mask_ratio=0.75, dropout=0.1,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

In [ ]:
# ── 5. Train ──────────────────────────────────────────────────────────────────
import time, torch.nn as nn

EPOCHS = 50
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

train_losses, val_losses = [], []
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    tl = 0.0
    for batch in train_loader:
        xyz = batch.to(device)
        loss, _, _ = model(xyz)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item()
    scheduler.step()

    # Validate
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for batch in val_loader:
            loss, _, _ = model(batch.to(device))
            vl += loss.item()

    tl /= len(train_loader)
    vl /= len(val_loader)
    train_losses.append(tl)
    val_losses.append(vl)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d}/{EPOCHS} | train={tl:.4f}  val={vl:.4f} | {time.time()-t0:.0f}s')

print(f'\nTraining complete in {time.time()-t0:.0f}s')
print(f'Loss improvement: {train_losses[0]:.4f} → {train_losses[-1]:.4f} ({(train_losses[0]-train_losses[-1])/train_losses[0]*100:.1f}% reduction)')

In [ ]:
# ── 6. Plot loss curve ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Chamfer Distance', linewidth=2)
plt.plot(val_losses,   label='Val Chamfer Distance',   linewidth=2, linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Chamfer Distance (lower = better)')
plt.title('PointMAE Self-Supervised Training on ModelNet10')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print('Loss curve saved to loss_curve.png')

In [ ]:
# ── 7. Visualise reconstruction ───────────────────────────────────────────────
from mpl_toolkits.mplot3d import Axes3D

model.eval()
sample = test_ds[0].unsqueeze(0).to(device)   # 1 point cloud

with torch.no_grad():
    _, pred, target = model(sample)

pred_pts   = pred.squeeze(0).reshape(-1, 3).cpu().numpy()
target_pts = target.squeeze(0).reshape(-1, 3).cpu().numpy()
input_pts  = sample.squeeze(0).cpu().numpy()

fig = plt.figure(figsize=(15, 5))

ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(*input_pts.T, s=0.5, c='gray', alpha=0.5)
ax1.set_title('Input Point Cloud', fontsize=11)
ax1.axis('off')

ax2 = fig.add_subplot(132, projection='3d')
ax2.scatter(*target_pts.T, s=2, c='orange', alpha=0.8)
ax2.set_title('Ground Truth\n(masked patches)', fontsize=11)
ax2.axis('off')

ax3 = fig.add_subplot(133, projection='3d')
ax3.scatter(*pred_pts.T, s=2, c='dodgerblue', alpha=0.8)
ax3.set_title('PointMAE Reconstruction\n(predicted patches)', fontsize=11)
ax3.axis('off')

plt.suptitle('PointMAE: Masked Patch Reconstruction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reconstruction.png', dpi=150)
plt.show()
print('Saved to reconstruction.png')

In [ ]:
# ── 8. Encoder features summary ──────────────────────────────────────────────
all_feats = []
with torch.no_grad():
    for batch in val_loader:
        all_feats.append(model.encode(batch.to(device)))
features = torch.cat(all_feats).cpu()

print('=' * 50)
print('  ENCODER OUTPUT SUMMARY')
print('=' * 50)
print(f'  Samples encoded   : {features.shape[0]}')
print(f'  Embedding dim     : {features.shape[1]}')
print(f'  Feature mean      : {features.mean():.4f}')
print(f'  Feature std       : {features.std():.4f}')
print(f'  Final train loss  : {train_losses[-1]:.4f}')
print(f'  Final val loss    : {val_losses[-1]:.4f}')
print('=' * 50)
print('Ready for downstream tasks (detection, segmentation).')